In [1]:
# Raw Nairaland
#         |
#         |
# Remove HTML
#         |
#         |
# Remove duplicate threads/comments
#         |
#         |
# Remove spam
#         |
#         |
# Remove phone/email
#         |
#         |
# Normalize whitespace
#         |
#         |
# Handle quotes

In [2]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


#

In [4]:
import json
import re
import unicodedata
from html.parser import HTMLParser


In [5]:
file_path = '/content/drive/MyDrive/YarnRAG/thread_data.jsonl'

In [6]:
with open(file_path, 'r') as file:
  for line in file:
      data = json.loads(line)
      break # Only load and print the first line
print(data)

{'category': 'politics', 'title': '2026 Osun Governorship Election Results Update (Photos)', 'link': 'https://www.nairaland.com/8729106/2026-osun-governorship-election-results', 'content': 'Follow this thread for all results from polling units in OSUN as they are been released', 'comments': ['More results as APC and Accord fight against each other', 'See more results as they are pumping', 'ADC-9APC-88Accord-51Our Lady School Modakeke', 'More results and keep can as I will post every results here', 'See even more results from polling units as they are made available to us', 'The mergin is very close. APC really want to take over Osun State.', 'More results coming and it seems Adeleke in early lead', 'Everywhere is silent No wonder 🤣', "This is bad news for Adeleke..Any governor who loses his capital loses his re-election..These released results are all on Adeleke's stronghold...It's not going well for him..cuz his margins are very little", 'See more results from polling units across the

In [7]:
# print(f"category{data["category"]}")
# print(f"title: {data["title"]}")
# print(f"link: {data["link"]}")
# print(f"content: {data["content"]}")
# for comment in data["comments"]:
#   print(comment["comment_number"])
#   print(comment["comment"])
#   print()

### General Text cleaning  
  
NB: Converting emails, phone nums and web links into placeholder tags preserves the structural context and sematic meaning of text during text preprocessing

In [8]:
# ---------------------------
# 1. Remove invisible unicode
# ---------------------------

def clean_unicode(text: str) -> str:
    """
    Remove invisible unicode characters while keeping Nigerian characters.
    """

    text = unicodedata.normalize("NFKC", text)

    # Remove zero-width and directional characters
    text = re.sub(r'[\u200b-\u200f\u202a-\u202e]', '', text)

    return text

In [9]:
# ---------------------------
# 2. Normalize whitespace
# ---------------------------

def clean_whitespace(text: str) -> str:
    """
    Remove excessive spaces and broken formatting.
    """

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [10]:
# ---------------------------
# 3. Remove HTML artifacts
# ---------------------------

class _HTMLTextExtractor(HTMLParser):
    """Extract visible text without relying on a fragile regex."""
    def __init__(self):
        super().__init__()
        self.parts = []
    def handle_data(self, data: str) -> None:
        self.parts.append(data)

def remove_html(text: str) -> str:
    parser = _HTMLTextExtractor()
    parser.feed(text)
    parser.close()
    return " ".join(parser.parts)


In [11]:
# ---------------------------
# 4. Remove URLs
# ---------------------------

def replace_links(text: str) -> str:
    """
    Replace links instead of deleting them.
    """

    text = re.sub(
        r'https?://\S+|www\.\S+',
        '[LINK]',
        text
    )

    return text

In [12]:
# ---------------------------
# 5. Redact emails and usernames
# ---------------------------

EMAIL_RE = re.compile(r"(?<![\w.+-])[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}(?![\w.-])")
MENTION_RE = re.compile(r"(?<!\w)@[A-Za-z0-9_]{2,30}\b")  # like @.....

# Nairaland author prefixes are emitted as `username:` at the start of a comment.
# The lookahead avoids stripping all-uppercase headings such as `LGA:`.
AUTHOR_PREFIX_RE = re.compile(r"^(?=[A-Za-z0-9_]{1,30}:)(?=[^:]*[a-z0-9_])[A-Za-z0-9_]+:\s*")

def redact_usernames(text: str, *, is_comment: bool = False) -> str:
    if is_comment:
        text = AUTHOR_PREFIX_RE.sub("", text)
    return MENTION_RE.sub("[USERNAME]", text)

def remove_emails(text: str) -> str:
    return EMAIL_RE.sub("[EMAIL]", text)


In [13]:
# ---------------------------
# 6. Redact Nigerian phone numbers
# ---------------------------

# Local: 08012345678 (0 + 10 digits); international: +2348012345678 (234 + 10 digits).
PHONE_RE = re.compile(r"(?<!\d)(?:\+?234[\s.-]?|0)(?:[\s.-]?\d){10}(?!\d)")

def remove_phone_numbers(text: str) -> str:
    return PHONE_RE.sub("[PHONE]", text)


In [14]:
# ---------------------------
# 7. Remove high-confidence spam only
# ---------------------------

SPAM_RE = re.compile(r"\b(?:join (?:my )?whatsapp|drop your number|check my signature|betting odds|download now)\b", re.IGNORECASE)

def is_spam(text: str) -> bool:
    return bool(SPAM_RE.search(text))


In [15]:
# ---------------------------
# 8. Main cleaner
# ---------------------------

def clean_text(text: str, *, is_comment: bool = False) -> str | None:
    if not isinstance(text, str) or not text.strip():
        return None
    text = clean_unicode(text)
    text = remove_html(text)
    text = replace_links(text)
    text = remove_emails(text)
    text = remove_phone_numbers(text)
    text = redact_usernames(text, is_comment=is_comment)
    text = clean_whitespace(text)
    if len(text.split()) < 3 or is_spam(text):
        return None
    return text


In [16]:
# ---------------------------
# 9. Deduplicate comments
# ---------------------------

def deduplicate_comments(comments: list[str]) -> list[str]:
    """Keep first occurrences while preserving discussion order."""
    seen = set()
    unique_comments = []
    for comment in comments:
        key = comment.casefold()
        if key not in seen:
            seen.add(key)
            unique_comments.append(comment)
    return unique_comments


### Main cleaning Process

In [17]:
# print(f"category{data["category"]}")
# print(f"title: {data["title"]}")
# print(f"link: {data["link"]}")
# print(f"content: {data["content"]}")
# for comment in data["comments"]:
#   print(comment["comment_number"])
#   print(comment["comment"])
#   print()

In [18]:
def process_thread_data(content: str, comments: list[str]) -> tuple[str | None, list[str]]:
    """Clean a thread without retaining comment-author identifiers."""
    cleaned_content = clean_text(content)
    cleaned_comments = [
        cleaned
        for item in comments
        if (cleaned := clean_text(item, is_comment=True)) is not None
    ]
    return cleaned_content, deduplicate_comments(cleaned_comments)


In [19]:
def format_text(title: str, content: str | None, comments: list[str]) -> str:
    """Create stable RAG text; never serialize Python None into the corpus."""
    main_post = content or "[NO USABLE MAIN POST]"
    comments_text = "\n\n".join(comments) if comments else "[NO USABLE COMMENTS]"
    return f"Thread Title: {title}\n\nMain Post: {main_post}\n\nComments:\n\n{comments_text}"


Cleans the dataset and selects a subset (600 threads) from them

In [21]:
TARGET_CATEGORIES = ("politics", "education", "sports", "jokes", "romance", "tech")
MAX_THREADS_PER_CATEGORY = 100
subset = []
category_counts = dict.fromkeys(TARGET_CATEGORIES, 0)

with open(file_path, encoding="utf-8") as file:
    for line in file:
        data = json.loads(line)
        category = data["category"]

        if category_counts[category] >= MAX_THREADS_PER_CATEGORY:
            continue

        title = clean_text(data["title"])
        cleaned_content, cleaned_comments = process_thread_data(data["content"], data["comments"])

        if not title or (cleaned_content is None and not cleaned_comments):
            continue

        subset.append({"source": "nairaland", "title": title, "category": category, "link": data["link"],
                       "main_content": format_text(title, cleaned_content, cleaned_comments),
                       "document_type": "discussion"})

        category_counts[category] += 1

print(f"Number of cleaned threads: {len(subset)}")
print(category_counts)


Number of cleaned threads: 600
{'politics': 100, 'education': 100, 'sports': 100, 'jokes': 100, 'romance': 100, 'tech': 100}


### Saving the cleaned data

In [22]:
from google.colab import files

In [23]:
output_filename = "cleaned_threads.json"

In [24]:
with open(output_filename, "w") as f:
  json.dump(subset, f, indent=4)

In [25]:
files.download(output_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>